<a href="https://colab.research.google.com/github/hania-sajjad/WEEK-6-TASK/blob/main/notebooks/week6_advanced_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6: Advanced Natural Language Processing (NLP)

## BBC News Analysis using Named Entity Recognition, Topic Modeling, and Transfer Learning


---

## Project Overview

Natural Language Processing (NLP) enables computers to understand, analyze, and derive meaningful insights from human language. In this project, three advanced NLP techniques are applied to the BBC News dataset to explore different aspects of text analysis.

The project consists of three major components:

- **Named Entity Recognition (NER):** Extract and analyze entities such as people, organizations, and locations from news articles.
- **Topic Modeling:** Discover hidden themes across the collection of articles using an unsupervised learning approach.
- **Transfer Learning:** Classify news articles into predefined categories using a pre-trained transformer model and compare its performance with a traditional machine learning baseline.

These techniques represent a complete NLP pipeline commonly used in applications such as news aggregation, media analytics, business intelligence, and information retrieval.

In [5]:
# Import Libraries

# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text Processing
import re
import string
import nltk

# spaCy for Named Entity Recognition
import spacy
from collections import Counter

# Topic Modeling
#!pip install gensim #not available by default
import gensim
from gensim import corpora
from gensim.models import CoherenceModel

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score
)

# Transformers
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

# PyTorch
import torch

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Set visualization style
sns.set_style("whitegrid")

print("Libraries imported successfully!")

Libraries imported successfully!


# Download Required NLP Resources

Several NLP libraries require additional language resources before they can be used.

In this section, the required datasets, tokenizers, stopwords, and language models are downloaded. These resources will support text preprocessing, Named Entity Recognition, and topic modeling throughout the project.

In [6]:
# Download NLTK resources
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# Download spaCy English model
#!python -m spacy download en_core_web_sm

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# Load the BBC News Dataset

The BBC News dataset is loaded into a Pandas DataFrame for analysis.

After loading the dataset, we will inspect its structure, verify that it has been read correctly, and ensure that it is suitable for further preprocessing and analysis.

In [17]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/Data/bbc_news.csv")

print("Dataset loaded successfully!\n")

print(f"Dataset Shape: ")
print(df.shape)

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully!

Dataset Shape: 
(42115, 5)


,title,pubDate,guid,link,description
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as..."
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...
3,Manchester Arena bombing: Saffie Roussos's par...,"Mon, 07 Mar 2022 00:05:40 GMT",https://www.bbc.co.uk/news/uk-60579079,https://www.bbc.co.uk/news/uk-60579079?at_medi...,The parents of the Manchester Arena bombing's ...
4,Ukraine conflict: Oil price soars to highest l...,"Mon, 07 Mar 2022 08:15:53 GMT",https://www.bbc.co.uk/news/business-60642786,https://www.bbc.co.uk/news/business-60642786?a...,Consumers are feeling the impact of higher ene...


In [18]:
df.info()
df.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42115 entries, 0 to 42114
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        42115 non-null  object
 1   pubDate      42115 non-null  object
 2   guid         42115 non-null  object
 3   link         42115 non-null  object
 4   description  42115 non-null  object
dtypes: object(5)
memory usage: 1.6+ MB


Index(['title', 'pubDate', 'guid', 'link', 'description'], dtype='object')